# 04 — Head-to-head performance evaluation

**Study:** Evaluating Protein Language Models for Function-Based Screening of Heme-Binding Proteins Against Traditional Sequence-Search Methods

**Question for this notebook:** How well do BLASTp and whole-protein ESM-2 representations rank heme binders above controls, especially among the natural positives without significant BLASTp matches?

**Inputs:** `blastp_matched_length_results.csv` from notebook 02 and `esm2_embedding_method_scores.csv` from notebook 03. Upload them **one at a time** if prompted.

**Analyses actually performed in Phase 1:** ROC AUC for all positives and each positive subgroup; protein-length check; focused analysis of the seven natural positives with BLASTp E-value > 0.01; and observed detection with one of 20 controls flagged.

The notebook reproduces the original Phase 1 analyses without introducing a new model or screening method.

In [ ]:
# Read results from notebooks 02 and 03 (without rerunning either method).
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from IPython.display import display

# Find each file in the working directory or in a sibling results folder.
def find_csv(name):
    for path in [Path(name), Path("../results") / name, Path("results") / name]:
        if path.exists():
            return path
    return None

# Upload a single file at a time if not already present in Colab.
def load_csv(name):
    path = find_csv(name)
    if path is None:
        from google.colab import files
        print("Upload", name)
        uploaded = files.upload()
        if name not in uploaded:
            raise ValueError("Please select the required file: " + name)
        path = Path(name)
    return pd.read_csv(path)

# Load the BLASTp matched-length results and our consolidated ESM-2 results.
blast = load_csv("blastp_matched_length_results.csv")
esm = load_csv("esm2_embedding_method_scores.csv")

# Require one unique row per test protein in each result file.
assert len(blast) == len(esm) == 50, "Each method should have 50 test records."
assert blast["record_id"].is_unique and esm["record_id"].is_unique
assert set(blast["record_id"]) == set(esm["record_id"]), "Test protein IDs differ."

# Merge our scores by stable protein record ID; do not rely on row order.
method_columns = [column for column in esm.columns if column.endswith("_similarity")]
comparison = blast.merge(
    esm[["record_id", "sequence_length", "length_used"] + method_columns],
    on="record_id", how="inner", validate="one_to_one"
)

# A protein with no reported BLASTp hit receives a best-bit-score of zero.
comparison["bitscore"] = comparison["bitscore"].fillna(0)

# Use the dataset's heme-binding positive/negative labels for ranking analyses.
comparison["label"] = comparison["heme_binding_label"].str.lower().eq("positive").astype(int)
print("Matched 50 test proteins across the two experiments.")

In [ ]:
# Compare the same continuous per-protein scores using AUC.
scoring_methods = {
    "BLASTp best bit score": "bitscore",
    "ESM-2 Layer 6 mean": "layer6_mean_similarity",
    "ESM-2 Layer 6 CLS": "layer6_cls_similarity",
    "ESM-2 Layer 3 mean": "layer3_mean_similarity",
    "ESM-2 Layer 3 max": "layer3_max_similarity",
}

# Collect ROC AUC scores for transparent ranked comparisons.
summary_rows = []
for group in ["ALL", "AI_DESIGNED_POSITIVE", "NATURAL_POSITIVE"]:
    # Each subgroup comparison keeps the 20 negative controls unchanged.
    if group == "ALL":
        block = comparison
    else:
        block = comparison[comparison["dataset_group"].isin([group, "NEGATIVE_CONTROL"])]

    # Calculate the metrics for each of the recorded methods.
    for method, column in scoring_methods.items():
        summary_rows.append({
            "group": group,
            "method": method,
            "n_tested": len(block),
            "roc_auc": roc_auc_score(block["label"], block[column]),
        })

# Save the full numeric results for the paper.
auc_table = pd.DataFrame(summary_rows)
print("ROC AUC BY METHOD AND TEST GROUP")
display(auc_table.pivot(index="method", columns="group", values="roc_auc").round(3))
auc_table.to_csv("phase1_auc_summary.csv", index=False)

# Save a single per-protein sheet containing both BLASTp and ESM-2 scores.
comparison.to_csv("phase1_matched_comparison.csv", index=False)

In [ ]:
# Examine whether protein length is associated with ESM-2 Layer-3 max-pooling scores.
print("PROTEIN LENGTHS BY GROUP")
display(comparison.groupby("dataset_group")["length_used"].agg(["count", "min", "median", "max"]))

# Spearman correlation compares the rankings of length and similarity.
length_rows = []
for group in ["ALL", "AI_DESIGNED_POSITIVE", "NATURAL_POSITIVE", "NEGATIVE_CONTROL"]:
    block = comparison if group == "ALL" else comparison[comparison["dataset_group"] == group]
    correlation = block["length_used"].corr(block["layer3_max_similarity"], method="spearman")
    length_rows.append({"group": group, "spearman_length_vs_max": correlation})

# Length-only AUC answers whether sequence length by itself ranks our positives above controls.
for row in length_rows:
    group = row["group"]
    if group == "NEGATIVE_CONTROL":
        row["length_only_auc"] = np.nan
        continue
    block = comparison if group == "ALL" else comparison[
        comparison["dataset_group"].isin([group, "NEGATIVE_CONTROL"])
    ]
    row["length_only_auc"] = roc_auc_score(block["label"], block["length_used"])

length_table = pd.DataFrame(length_rows)
print("LENGTH ANALYSIS")
display(length_table.round(3))
length_table.to_csv("phase1_length_check.csv", index=False)

In [ ]:
# Identify natural positives for which BLASTp did not report E <= 0.01.
missed_mask = (
    comparison["dataset_group"].eq("NATURAL_POSITIVE") &
    comparison["evalue"].fillna(float("inf")).gt(0.01)
)
missed_ids = set(comparison.loc[missed_mask, "record_id"])
print("Natural positives with no significant BLASTp match:")
print(comparison.loc[missed_mask, "protein_name"].tolist())
print("Number:", len(missed_ids))

# Compare this focused natural-positive subset with the same 20 controls.
focused = comparison[
    comparison["record_id"].isin(missed_ids) |
    comparison["dataset_group"].eq("NEGATIVE_CONTROL")
].copy()

# Use the two whole-protein methods examined in the pilot.
focused_methods = {
    "BLASTp best bit score": "bitscore",
    "ESM-2 Layer 3 mean": "layer3_mean_similarity",
    "ESM-2 Layer 3 max": "layer3_max_similarity",
}

# The in-sample operating point allows at most one of the 20 controls above the cutoff.
rows = []
for method, column in focused_methods.items():
    negatives = focused.loc[focused["label"] == 0, column].sort_values(ascending=False)
    cutoff = float(negatives.iloc[1])
    predictions = focused[column] > cutoff
    detected = int((predictions & focused["label"].eq(1)).sum())
    flagged = int((predictions & focused["label"].eq(0)).sum())
    auc = roc_auc_score(focused["label"], focused[column])
    rows.append({
        "method": method, "roc_auc": auc, "cutoff": cutoff,
        "positives_detected": detected, "positives_total": len(missed_ids),
        "controls_flagged": flagged, "controls_total": 20,
    })

# Show the same headline metrics from our seven-protein pilot analysis.
operating_points = pd.DataFrame(rows)
display(operating_points.round(3))
operating_points.to_csv("phase1_operating_point_summary.csv", index=False)

# Save the complete focused subset and the seven positive protein details.
focused.to_csv("blastp_vs_esm2_seven_proteins.csv", index=False)
display(focused.loc[focused["label"] == 1, [
    "protein_name", "bitscore", "evalue",
    "layer3_mean_similarity", "layer3_max_similarity",
]])

In [ ]:
# Download compact, reviewable result tables for the GitHub results folder.
from google.colab import files
files.download("phase1_auc_summary.csv")
files.download("phase1_matched_comparison.csv")
files.download("phase1_length_check.csv")
files.download("phase1_operating_point_summary.csv")
files.download("blastp_vs_esm2_seven_proteins.csv")

### Original pilot results to compare against

| Comparison | BLASTp best bit score | ESM-2 Layer 3 mean | ESM-2 Layer 3 max |
|---|---:|---:|---:|
| All positives vs controls: ROC AUC | 0.905 | 0.542 | 0.753 |
| Natural positives vs controls: ROC AUC | 0.715 | 0.530 | 0.655 |
| Seven natural positives without significant BLASTp hits: ROC AUC | 0.593 | 0.486 | 0.586 |
| Seven-protein subset: detected with 1/20 controls flagged | 0/7 | 1/7 | 1/7 |

The seven-protein subset was selected by BLASTp's significance rule; its analysis complements, rather than replaces, the full-set comparison. These are the recorded numerical findings from our pilot, while the code above recalculates them from the submitted CSVs.